---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-16: Hands-on Understanding of Tokenization</h1>

# Learning agenda of this notebook

1. What is Tokenization and Why we need it?
2. Tokenization in the Early NLP Era (`string`, `re`, `nltk`, `spacy`)
    - String-based Tokenization using `string.split()` Method
    - Regex (re) Tokenization with `re.split()` Method
    - NLTK Tokenization using NLTK (rule-based linguistic tokenizers)
    - spaCy Tokenization with spaCy
3. Pre-Trained LLM Tokenizers
4. Online Tokenization Tools
5. Hands on Examples of using Hugging Face Tokenizers for Base Models
6. Hands on Examples of using Hugging Face Tokenizers for Chat Models
7. Comparing Trained LLM Tokenizers

# <span style='background :lightgreen' >1. What is Tokenization and Why we need it?</span>
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Tokenization is a process of splitting text into meaningful segments called tokens. It can be character level, subword level, word level (unigram), two word level (bigram), three word level (trigram), and sentence level.</h3>


<h2 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The Raw text (prompt) is first split into smaller pieces called <b>tokens</b>, which are then converted into integer IDs that becomes the model input.</h2>

## Pre-processing: Pre-LLM Era vs LLM Era

#### Pre-LLM Era
- **Pipeline:** `Raw Text → Lowercasing → Noise Removal → Stemming/Lemmatization → Tokenization → Model`
- Heavy pre-processing was **mandatory** because models like Bag-of-Words and TF-IDF had no ability to learn context — they treated every unique word as an isolated, unrelated token
- "Running", "runs", and "ran" were seen as 3 completely different words, bloating the vocabulary unnecessarily
- Stemming and lemmatization were used to convert a word to its base/root form by removing its prefix and suffix:
    - **Stemming** blindly chops the prefix/suffix without checking if the result is a real word
    - **Lemmatization** uses a lexical dictionary called WordNet to ensure the reduced form is a valid dictionary word
    - Both are available in NLTK: `from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer`
- Interested  Students can watch following two videos on Text Pre-Processing:
    - **[Text Pre-Processing - I by Arif](https://www.youtube.com/watch?v=8PTEaCI5GTU&list=PL7B2bn3G_wfANUMx-dOv_qACGKchtDWum&index=2)**
    - **[Text Pre-Processing - II by Arif](https://www.youtube.com/watch?v=OIria2ylIbY&list=PL7B2bn3G_wfANUMx-dOv_qACGKchtDWum&index=3)**

#### LLM Era
- **Pipeline:** `Raw Text → Tokenization (via BPE / WordPiece) → Token IDs → Model`
- Heavy pre-processing is no longer needed, because LLMs learn during training that "run", "running", and "ran" are related, without being told explicitly
- Stemming and lemmatization are intentionally skipped because they discard grammatical information (tense, plurality) that transformers actively rely on
- Subword tokenizers like Byte Pair Encoding (BPE) and WordPiece handle morphology automatically by splitting unknown or rare words into meaningful pieces — e.g., `"unhappiness" → ["un", "##happi", "##ness"]` — without throwing away information

<img align=right src="../images/tokenization44.png" width="1000">

# Tokenization Levels
* **Word-Level Tokenization:**
  * Individual words are treated as separate tokens. This approach was common in earlier NLP systems (e.g., word2vec) but is now less widely used.
  * **Advantages:**
      * Capture meanings of words
      * Simple, intuitive, and easy to understand.
      * Suitable for language modeling
  * **Limitations:**
      * Cannot handle out-of-vocabulary (OOV) words
      * Treating each word as a separate token leads to very large vocabularies with many similar word variants (e.g., apology, apologize, apologetic, apologist).
* **Character-Level Tokenization:**
  * Each character—including letters, spaces, and punctuation—is treated as an individual token.
  * Because it operates on raw characters, it can handle unseen words naturally. However, it produces long token sequences, making modeling more difficult. For example, “play” becomes `p-l-a-y` rather than a single subword token.
  * **Advantages:**
      * Small vocabulary size.
      * No out-of-vocabulary  (OOV) problem.
  * **Limitations:**
      * Captures no contextual meanings.
      * Generates much longer sequences.
      * Increase computational load.
* **Subword-Level Tokenization:**
  * Words are broken into smaller, meaningful units (subwords) that balance vocabulary size, generalization ability, and linguistic structure.
  * This approach combines full and partial words. It handles new words gracefully by decomposing unfamiliar terms into smaller known pieces while keeping common words as single tokens.
  * Subword tokenization is more efficient than character-level methods, allowing far more text to fit within a transformer's fixed context window (e.g., subword tokens often average three characters each).

## Why to do Tokenization?
- For classification of a product review as positive or negative, we may need to count the number of positive/negative words in the text of that product review. For this we first need to tokenize the text of the product review. If the count of positive words is greater than the count of negative words, we say that it is a positive review and vice versa.
- Tokens are the basic building blocks of a document object.
- Everything that helps us understand the meaning of the text is derived from tokens and their relationship to one another.

# <span style='background :lightgreen' >2. Tokenization in the Early NLP Era (**string**, **re**, **nltk**, **spacy**)</span>
- Early NLP era use libraries that use **linguistic rule-based approaches**:
    - **String-based tokenization** splits text using simple string operations (`string.split()`), treating whitespace as token boundaries.
    - **Regex (re) tokenization** uses pattern-based rules to split text (`re.split()`), allowing control over punctuation, numbers, and custom token patterns.
    - **NLTK tokenization:** applies rule-based linguistic tokenizers that handle punctuation, contractions, and basic language nuances.
    - **SpaCy tokenization** uses a production-grade NLP pipeline with statistical and rule-based models to segment text into linguistically precise tokens.

In [1]:
str1="Learning is fun with Arif" 
str2="You should do your Ph.D in A.I!" 
str3="'A 7km Uber cab ride from Gulberg to Joher-Town will cost you $20" 
str4="You should've sent me an email at arif@pucit.edu.pk or vist http://www/arifbutt.me"
str5="Here's an example worth $100. I am 384400km away from earth's moon!" 

## (i) String-based Tokenization using `string.split()` Method
- The easiest way to tokenize is to use the `.split()` method of string class, which returns a list of strings. It splits a string into a list of strings at every occurrence of space character by default and discard empty strings from the result.
- You may pass a parameter `sep='i'` to split method to split at that specific character instead.
- It's limitation is that it do not  consider punctuation symbols as a separate token

In [1]:
str1="Learning is fun with Arif" 
str2="You should do your Ph.D in A.I!" 
str3="'A 7km Uber cab ride from Gulberg to Joher-Town will cost you $20" 
str4="You should've sent me an email at arif@pucit.edu.pk or vist http://www/arifbutt.me"
str5="Here's an example worth $100. I am 384400km away from earth's moon!" 
print(str1.split())
print(str2.split())
print(str3.split())
print(str4.split())
print(str5.split())

['Learning', 'is', 'fun', 'with', 'Arif']
['You', 'should', 'do', 'your', 'Ph.D', 'in', 'A.I!']
["'A", '7km', 'Uber', 'cab', 'ride', 'from', 'Gulberg', 'to', 'Joher-Town', 'will', 'cost', 'you', '$20']
['You', "should've", 'sent', 'me', 'an', 'email', 'at', 'arif@pucit.edu.pk', 'or', 'vist', 'http://www/arifbutt.me']
["Here's", 'an', 'example', 'worth', '$100.', 'I', 'am', '384400km', 'away', 'from', "earth's", 'moon!']


> <font color=green> Observe the outputs, and note which limitations out of following exist in `string.split()` method
>- **Handling prefixes:** Characters at the beginning, e.g., `Dr`, `Rs`, `$`, `(`, `"` etc
>- **Handling suffixes:** Characters at the end, e.g., `!`, `.`, `)`, `"` etc
>- **Handling infixes:** Characters in-between, e.g., `-`, `--`, `/`, `...` etc
>- **Handling exceptions:** Special rules to split a string into tokens, e.g., in `A.I!` the exclamation mark need to be separated. Special rules NOT to split a string into tokens, e.g., in `A.I` the `A` and `L` need not be separated.

## (ii) Regex (re) Tokenization with `re.split()` Method
- Another way to tokenize is to use the `re.split()` method of Regular Expression class, that splits the source string by the occurrences of a pattern, returning a list containing the resulting substrings.
- The `\W` split the token on occurrence of any non-alphanumeric character, i.e., everything other than ( (a-z, A-Z), digits (0-9), and underscore (_)), which include:
    - Spaces, tabs, newlines
    - Punctuation: .,!?;:"'()[]{}
    - Special symbols: @#$%^&*+-=<>/|~`
- The `+` is a quantifier that means "one or more" of the preceding character
- So in summary `\W+` will split on punctuation, whitespace, and special symbols
- Interested  Students can watch following two videos on Regular Expression:
    - **[Regular Expression - I by Arif](https://www.youtube.com/watch?v=DhQ-kc6FPVk&list=PL7B2bn3G_wfAs3C49i12i_rblzvuU1dFN&index=20)**
    - **[Regular Expression - II by Arif](https://www.youtube.com/watch?v=3J62z5aGTQc&list=PL7B2bn3G_wfAs3C49i12i_rblzvuU1dFN&index=21)**

In [2]:
import re
str1="Learning is fun with Arif" 
str2="You should do your Ph.D in A.I!" 
str3="'A 7km Uber cab ride from Gulberg to Joher-Town will cost you $20" 
str4="You should've sent me an email at arif@pucit.edu.pk or vist http://www/arifbutt.me"
str5="Here's an example worth $100. I am 384400km away from earth's moon!" 

pattern = re.compile(r'\W+') # split the token on any non-alphanumeric character
print(pattern.split(str1))
print(pattern.split(str2))
print(pattern.split(str3))
print(pattern.split(str4))
print(pattern.split(str5))

['Learning', 'is', 'fun', 'with', 'Arif']
['You', 'should', 'do', 'your', 'Ph', 'D', 'in', 'A', 'I', '']
['', 'A', '7km', 'Uber', 'cab', 'ride', 'from', 'Gulberg', 'to', 'Joher', 'Town', 'will', 'cost', 'you', '20']
['You', 'should', 've', 'sent', 'me', 'an', 'email', 'at', 'arif', 'pucit', 'edu', 'pk', 'or', 'vist', 'http', 'www', 'arifbutt', 'me']
['Here', 's', 'an', 'example', 'worth', '100', 'I', 'am', '384400km', 'away', 'from', 'earth', 's', 'moon', '']


> <font color=green> Observe the outputs, and note which limitations out of following exist in `re.split()` method (Moreover, you need to write different regular expression for different scenarios)
>- **Handling prefixes:** Characters at the beginning, e.g., `Dr`, `Rs`, `$`, `(`, `"` etc
>- **Handling suffixes:** Characters at the end, e.g., `!`, `.`, `)`, `"` etc
>- **Handling infixes:** Characters in-between, e.g., `-`, `--`, `/`, `...` etc
>- **Handling exceptions:** Special rules to split a string into tokens, e.g., in `A.I!` the exclamation mark need to be separated. Special rules NOT to split a string into tokens, e.g., in `A.I` the `A` and `I` need not be separated.

## (iii) NLTK Tokenization using NLTK (rule-based linguistic tokenizers)
- **NLTK** stands for Natural Language Toolkit (https://www.nltk.org/). This is a suite of libraries and programs for statistical natural language processing for English language. Suppport of other languages like Spanish or French are not supported as extensively. It was released in 2001, and is available for Windows, Mac OS X, and Linux.. 
- NLTK provides easy-to-use interfaces to over 50 corpora and lexical resources such as WordNet, along with a suite of text processing libraries for classification, tokenization, stemming, tagging, parsing, and semantic reasoning.
- It is a string processing library, i.e., you give a string as input and get a string as output. There are. different tokenizer available in nltk:
    - `nltk.tokenize.sent_tokenize(str)` for sentence tokenization
    - `nltk.tokenize.word_tokenize(str)` for word tokenization
    - `nltk.tokenize.treebank.TreebankWordTokenizer(str)`
- After you install nltk using `!uv add nltk`, you also need to download some tokenizer model for sentence boundary detection. You can download `punkt_tab` using the `nltk.download('punkt_tab')` command.
- The `punkt_tab` uses the `Punkt` algorithm, which is an unsupervised machine learning approach that learns abbreviations, collocations, and sentence starters from training data. The `tab` suffix indicates this is a newer, more efficient tabular format version of the original `Punkt` tokenizer. On my mac machine, I have already downloaded it inside the `/Users/arif/nltk_data/tokenizers/punkt_tab`

> <font color=red> Be watchful of the size of `punkt_tab` file, which is saved on your disk and occupies ~ 5MiBs

In [3]:
import nltk
from nltk.tokenize import word_tokenize
print(nltk.__version__)

str1="Learning is fun with Arif" 
str2="You should do your Ph.D in A.I!" 
str3="'A 7km Uber cab ride from Gulberg to Joher-Town will cost you $20" 
str4="You should've sent me an email at arif@pucit.edu.pk or vist http://www/arifbutt.me"
str5="Here's an example worth $100. I am 384400km away from earth's moon!" 

print(word_tokenize(str1))
print(word_tokenize(str2))
print(word_tokenize(str3))
print(word_tokenize(str4))
print(word_tokenize(str5))

3.9.3
['Learning', 'is', 'fun', 'with', 'Arif']
['You', 'should', 'do', 'your', 'Ph.D', 'in', 'A.I', '!']
["'", 'A', '7km', 'Uber', 'cab', 'ride', 'from', 'Gulberg', 'to', 'Joher-Town', 'will', 'cost', 'you', '$', '20']
['You', 'should', "'ve", 'sent', 'me', 'an', 'email', 'at', 'arif', '@', 'pucit.edu.pk', 'or', 'vist', 'http', ':', '//www/arifbutt.me']
['Here', "'s", 'an', 'example', 'worth', '$', '100', '.', 'I', 'am', '384400km', 'away', 'from', 'earth', "'s", 'moon', '!']


> <font color=green> Observe the outputs, and note which limitations out of following exist in NLTK Tokenizer
>- **Handling prefixes:** Characters at the beginning, e.g., `Dr`, `Rs`, `$`, `(`, `"` etc
>- **Handling suffixes:** Characters at the end, e.g., `!`, `.`, `)`, `"` etc
>- **Handling infixes:** Characters in-between, e.g., `-`, `--`, `/`, `...` etc
>- **Handling exceptions:** Special rules to split a string into tokens, e.g., in `A.I!` the exclamation mark need to be separated. Special rules NOT to split a string into tokens, e.g., in `A.I` the `A` and 'I` need not be separated.

## (iv) spaCy Tokenization with spaCy
- **spaCy** (https://spacy.io/) is an open-source Natural Language Processing library designed to handle NLP tasks with the most efficient and state of the art algorithm, released in 2015.
- spaCy uses a production-grade NLP pipeline with statistical and rule-based models to segment text into linguistically precise tokens.
- Spacy support many languages (over 65) where you can perform tokenizing, however, for this other than importing spacy, you have to load the appropriate library using spacy.load() method. But before that make sure you have downloaded the model in your system.
- After you install spacy using `!uv add spacy`, you also need to download some spacy model for English language. Spacy comes with pretrained models and pipelines for different languages like  `en_core_web_sm`, `en_core_web_md` and others.
- You can download `en_core_web_sm` by giving following command:
```
!pip install en_core_web_sm
```
- The usage is simple: The `nlp = spacy.load('en_core_web_sm')` line of code loads the a pre-trained English language model and returns a complete NLP pipeline object, which is a callable pipeline that can process text and contains tokenizer, parts-of-speech tagger, named entity recognition, lemmatizer etc

> <font color=red> Be watchful of the size of `en_core_web_sm` file, which is save on your disk and occupies ~ 50 MiBs

In [5]:
import spacy

#nlp = spacy.load("en_core_web_sm")
nlp = spacy.load("/opt/anaconda3/lib/python3.12/site-packages/en_core_web_sm/en_core_web_sm-3.8.0")
print("Model loaded successfully!")

Model loaded successfully!


In [6]:
str1="Learning is fun with Arif" 
str2="You should do your Ph.D in A.I!" 
str3="'A 7km Uber cab ride from Gulberg to Joher-Town will cost you $20" 
str4="You should've sent me an email at arif@pucit.edu.pk or vist http://www/arifbutt.me"
str5="Here's an example worth $100. I am 384400km away from earth's moon!" 

print([token.text for token in nlp(str1)])
print([token.text for token in nlp(str2)])
print([token.text for token in nlp(str3)])
print([token.text for token in nlp(str4)])
print([token.text for token in nlp(str5)])

['Learning', 'is', 'fun', 'with', 'Arif']
['You', 'should', 'do', 'your', 'Ph', '.', 'D', 'in', 'A.I', '!']
["'", 'A', '7', 'km', 'Uber', 'cab', 'ride', 'from', 'Gulberg', 'to', 'Joher', '-', 'Town', 'will', 'cost', 'you', '$', '20']
['You', 'should', "'ve", 'sent', 'me', 'an', 'email', 'at', 'arif@pucit.edu.pk', 'or', 'vist', 'http://www', '/', 'arifbutt.me']
['Here', "'s", 'an', 'example', 'worth', '$', '100', '.', 'I', 'am', '384400', 'km', 'away', 'from', 'earth', "'s", 'moon', '!']


> <font color=green> Note that spacy has kept the email as a single token, while nltk separated it.</font><br>
> <font color=green> However, spacy also failed to properly tokenize the URL :(</font><br>
> <font color=green> Observe the outputs, and note which limitations out of following exist in NLTK Tokenizer
>- **Handling prefixes:** Characters at the beginning, e.g., `Dr`, `Rs`, `$`, `(`, `"` etc
>- **Handling suffixes:** Characters at the end, e.g., `!`, `.`, `)`, `"` etc
>- **Handling infixes:** Characters in-between, e.g., `-`, `--`, `/`, `...` etc
>- **Handling exceptions:** Special rules to split a string into tokens, e.g., in `L.A!` the exclamation mark need to be separated. Special rules NOT to split a string into tokens, e.g., in `L.A` the `L` and 'A` need not be separated.

# <span style='background :lightgreen' >3. Pre-Trained LLM Tokenizers</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">A pre-trained LLM tokenizer is a data-learned subword segmentation system, that breaks down text into smaller units called tokens, then looks up each token in a fixed vocabulary and maps it to a unique integer ID. These token IDs are what the language model consumes as input during pre-training, fine-tuning, and inference.</h3>

<div style="text-align:center;">
    <img src="../images/tokenizer99.png"
         style="max-width:1200px; width:100%; height:auto; display:inline-block;">
</div>

## How a Tokenizer is Trained?
- **Vocabulary and Tokenization:** What do you do first when you start learning a new language? You start acquiring its vocabulary, expanding it as you gain more proficiency in the language. Let’s define vocabulary here as: "All the words in a language that are understood by a specific person". The average native English speaker is said to have a vocabulary ranging between 20,000-35,000 words. Similarly, every language model has its own vocabulary, with most vocabulary sizes ranging anywhere between 5,000 to 500,000 tokens.
- Tokenization is the first and most essential step in any NLP workflow: transformer models cannot read raw text; they only understand sequences of numbers (token IDs).
- The transformers library provides tokenizers that are paired with each model, ensuring that text is preprocessed precisely the way the model expects.

## Three Main Categories of Tokenizers
### (i) BytePiece-Based Tokenizers (use frequency):
- Merges the most frequent pairs of characters step by step to create subwords, so rare words can still be represented.
- Example: The BPE tokenization of the word `"unhappiness"`: [`"un"`, `"h"`, `"appiness"`]. Note that the frequent character pairs like `"ha"` and `"pp"` get merged iteratively into subwords.
    - **GPT2Tokenizer** is used by GPT-2, GPT-Neo, GPT-J, CodeParrot: `from transformers import GPT2Tokenizer`
    - **RobertaTokenizer** is used by RoBERTa, DistilRoBERTa, BART, DeBERTa: `from transformers import RobertaTokenizer` 
    - **CodeGenTokenizer** is used by CodeGen, Salesforce code models: `from transformers import CodeGenTokenizer` 
    - **WhisperTokenizer** is used by Whisper speech recognition models: `from transformers import WhisperTokenizer`
    - **CLIPTokenizer** is used by vision-language models: `from transformers import CLIPTokenizer`
### (ii) WordPiece-Based Tokenizers (use liklihood):
- Breaks words into subwords from a fixed vocabulary to maximize text likelihood, ensuring unknown words can be split into known pieces. Best for masked language models like BERT and classification tasks, as it balances vocabulary size and semantic consistency.
- Example: The WordPiece tokenization of the word `"unhappiness"`: [`"un"`, `"##happy"`, `"##ness"`], Note that `##` indicates it’s a continuation of a known word piece from the vocabulary.
    - **BertTokenizer** is used by BERT, ClinicalBERT, SciBERT, BioBERT: `from transformers import BertTokenizer`
    - **ElectraTokenizer** is used by ELECTRA: `from transformers import ElectraTokenizer` 
### (iii) SentencePiece-Based Tokenizers
- Learns subwords directly from raw text without relying on spaces, making it suitable for multiple languages and text with no clear word boundaries. Best for multilingual or whitespace-agnostic tasks.
- Example: The SentencePiece tokenization of the sentence `"I love programming"`: [`"▁I"`, `"▁love"`, `"▁pro"`, `"gram"`, `"ming"`]. Note that  `▁` marks word boundaries; it can split any text (even languages with no spaces, like Japanese).
    - **T5Tokenizer** is used by T5, UL2, Flan-T5, mT5: `from transformers import T5Tokenizer`
    - **LlamaTokenizer** is used by LLaMA, LLaMA-2, Alpaca, Vicuna: `from transformers import LlamaTokenizer`

<div style="text-align:center;">
    <img src="../images/tokenization-new.png"
         style="max-width:1200px; width:100%; height:auto; display:inline-block;">
</div>


- **Special Tokens:** Some tokenizers use special tokens for unknown words and to denote the end of a text (useful in fine-tuning stage). Some of these special tokens can be:
  - `[BOS]` (beginning of sequence) marks the beginning of text
  - `[EOS]` (end of sequence) marks where the text ends (this is usually used to concatenate multiple unrelated texts, e.g., two different Wikipedia articles or two different books, and so on)
  - `[PAD]` (padding) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length)
  - `[UNK]` to represent words that are not included in the training set vocabulary (out-of-vocabulary words)




<div style="text-align:center;">
    <img src="../images/tokenizer35.png"
         style="max-width:1200px; width:100%; height:auto; display:inline-block;">
</div>



# <span style='background :lightgreen' >4. Online Tokenization Tools</span>
- **https://platform.openai.com/tokenizer:** OpenAI provides an interactive tokenizer web interface that lets you type/paste text and instantly see how it is broken into tokens for GPT-2, GPT-3, GPT-3.5, and GPT-4 models. It helps you estimate token counts, understand tokenization behavior, and plan prompt sizes before sending API requests.
- **https://tiktokenizer.vercel.app/:** A community-built tokenizer that uses OpenAI’s tiktoken library to show how different OpenAI models (GPT-3.5, GPT-4, GPT-4o, etc.) tokenize text. It supports live token counting, multiple model options, cost estimation, and a more interactive UI than the official tool.

>- Try observing how different model tokenizers breaks compound words, contractions, code, and even how it treats punctuation marks as separate tokens, by copy/pasting the following sample texts:
    - Demystifying Tokenization: The Building Bocks of Language AI
    - The quick brown fox jumps over the lazy dog (9 tokens, 43 characters)
    - Arif's favourite number is 3.1415926535897932384626433832795028842 (22 tokens, 66 characters)
    - OpenAI's GPT-4 is powerful (8 tokens, 26 characters)
    - asdfsomejkl;unknown34byettobc (8 tokens, 26 characters)

# <span style='background :lightgreen' >5. Hands on Examples of using Hugging Face Tokenizers for Base Models</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The transformers library has a set of Auto classes, like AutoConfig, AutoModel, and AutoTokenizer. The Auto classes are designed to automatically do the job for you. </h3>

### AutoTokenizer Class of Transformers Library
- Instead of importing  a specific tokernizer and then calling its `.from_pretrained()` method, it is better to import `Autotokenizer` and then calling its `.from_pretrained()` method as show in the code snippet below:
    - `AutoTokenizer` automatically selects the correct tokenizer class based on the model name.
    - `AutoTokenizer` examines the model's configuration file to determine which specific tokenizer class to instantiate and returns the appropriate tokernizer object.
- Instead of downloading a specific tokenizer, better approach is to use the `transformers.AutoTokenizer.from_pretrained()` method, to which you specify the model and it automatically download/load the pre-trained tokenizer for that specific model.
- `DistilGPT-2` uses Byte-Level BPE tokenizer, having a vocab of ~50K, with a word boundary marker of `Ġ`. The models that uses this are GPT-2-style decoder only architectures.
- When you run `tokenizer = AutoTokenizer.from_pretrained('distilgpt2')` for the first time, it will download and save the tokernizer files from Hugging Face Hub to your local machine's hard disk. Subsequently, the tokernizer is loaded instantly from local cache (no re-download)
    - macOS: /Users/[username]/.cache/huggingface/
    - Linux: /home/[username]/.cache/huggingface/
    - Windows: C:\Users\[username]\.cache\huggingface\
- **Note:** 
    - `AutoTokenizer.from_pretrained(...)` → downloads tokenizer only usually a few MiB containing files like tokenizer.json, tokenizer.model, special_tokens_map.json, tokenizer_config.json
> <font color=red> Be watchful of the size of these tokenizers you download

### Using `GPT2Tokenizer` or `AutoTokenizer`

In [7]:
from transformers import AutoTokenizer # AutoTokenizer automatically selects the correct tokenizer class based on the model name you provide.
from transformers import GPT2Tokenizer # Import the specific tokenizer class designed for GPT-2 models

tokenizer = AutoTokenizer.from_pretrained("gpt2")
#tokenizer = GPT2Tokenizer.from_pretrained("gpt2") 

# Print all the tokens along with their tokenIDs using `tokenizer.vocab`
print("Vocabulary Size:", tokenizer.vocab_size)      

# Breakdown the text into tokens
tokens = tokenizer.tokenize("I love transformers")
print("Tokens:", tokens)

# Breakdown the text and encode the tokens into token IDs
tokenIDs = tokenizer.encode("I love transformers")
print("Token IDs:", tokenIDs)

# Decode the tokenIDs back to text token
decodedTokens = tokenizer.decode(tokenIDs)
print("Decodedd TokenIDs:", decodedTokens)

Vocabulary Size: 50257
Tokens: ['I', 'Ġlove', 'Ġtransform', 'ers']
Token IDs: [40, 1842, 6121, 364]
Decodedd TokenIDs: I love transformers


>- The GPT-2 tokenizer does not store spaces as separate tokens.
>- It attaches the space to the beginning of the next token, and marks it with the symbol **Ġ**.

### Using `toktoken.get_encoding("gpt2")`

In [8]:
import tiktoken

# Load GPT-2 tokenizer and check size of its vocabulary 
tokenizer = tiktoken.get_encoding("gpt2")
print("Vocabulary Size:", tokenizer.n_vocab)

# Breakdown the text and encode the tokens into token IDs
token_ids = tokenizer.encode("I love transformers") 
print("Token IDs:", token_ids)

# Decode the tokenIDs back to text token
decoded_text = tokenizer.decode(token_ids)
print("Decoded Token IDs:", decoded_text)

Vocabulary Size: 50257
Token IDs: [40, 1842, 6121, 364]
Decoded Token IDs: I love transformers


# <span style='background :lightgreen' >6. Hands on Examples of using Hugging Face Tokenizers for Chat Models</span>

- Instruction-tuned models like `'meta-llama/Meta-Llama-3.1-8B-Instruct'`, `mistral-7b-instruct`, `flan-t5`, or `gpt-3.5-turbo` are fine-tuned further on instruction–response datasets to follow user directions naturally. These models are optimized for multi-turn dialogue, reasoning, and user-friendly interaction, making them ideal for chatbots and assistants.
- Some of the models have a variant that has been trained for use in Chats. These are typically labelled with the word "Instruct" at the end.
- They have been trained to expect prompts with a particular format that includes system, user and assistant prompts.There is a utility method `apply_chat_template` that will convert from the messages list format we are familiar with, into the right input prompt for this model.

## a. Downloading Hugging Face Tokenizer for `meta-llama/Meta-Llama-3.1-8B-Instruct`
- The `Meta-Llama-3.1-8B-Instruct`uses SentencePiece BPE tokenizer, having a vocab of ~128K, with a word boundary marker of `_`. The models that uses this are LlaMA encoder-decoder or decoder-only variants.
- In order to use a tokenizer, e.g., the fantastic Llama 3.1, Meta does require you to sign their terms of service. Follow the following steps:
    - Visit the model page: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B
    - Click the "Request access" button
    - Fill out the access request form
    - Wait for approval (can take hours to days)
- Visit https://huggingface.co/settings/gated-repos, in order to view the gated repositories that you have requested access to
> <font color=red> Be watchful of the size of these tokenizers you download

In [9]:
from transformers import AutoTokenizer

# Load GPT-2 tokenizer and check its vocabulary
tokenizer = AutoTokenizer.from_pretrained("gpt2") 
print("Vocabulary Size:", tokenizer.vocab_size)  # Can print all the tokens along with their tokenIDs using `tokenizer.vocab`

# Breakdown the text into tokens
tokens = tokenizer.tokenize("I love transformers")
print("Tokens:", tokens)

# Breakdown the text and encode the tokens into token IDs
tokenIDs = tokenizer.encode("I love transformers")
print("Token IDs:", tokenIDs)

# Decode the tokenIDs back to text token
decodedTokens = tokenizer.decode(tokenIDs)
print("Decodedd TokenIDs:", decodedTokens)

Vocabulary Size: 50257
Tokens: ['I', 'Ġlove', 'Ġtransform', 'ers']
Token IDs: [40, 1842, 6121, 364]
Decodedd TokenIDs: I love transformers


In [10]:
import transformers

# Load 'meta-llama/Meta-Llama-3.1-8B-Instruct' tokenizer 
llama_tokenizer = transformers.AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct')

print("Vocabulary Size:", llama_tokenizer.vocab_size)                         # The .vocab_size do not include the special tokens used by this instruct modelwhich for llama_tokenizer are 256
print("Added Vocabulary Size:", len(llama_tokenizer.get_added_vocab()))       # The added special tokens that are used by llama  instruct are 256

#print("Added Vocabulary:", llama_tokenizer.get_added_vocab()) 

Vocabulary Size: 128000
Added Vocabulary Size: 256


## Tokenizing Simple Text Strings

In [11]:
# Breakdown the text into tokens
tokens = llama_tokenizer.tokenize("I love transformers")
print("Tokens:", tokens)

# Breakdown the text and encode the tokens into token IDs
token_ids = llama_tokenizer.encode("I love transformers") 
print("Token IDs:", token_ids)

# Decode the tokenIDs back to text token
decoded_text = llama_tokenizer.decode(token_ids)
print("Decoded Token IDs:", decoded_text)

Tokens: ['I', 'Ġlove', 'Ġtransformers']
Token IDs: [128000, 40, 3021, 87970]
Decoded Token IDs: <|begin_of_text|>I love transformers


## Tokenizing Prompts specific for Chat Models
- The tokenizers for chat/instruct models are trained to expect prompts with a particular format that includes system, user and assistant prompts
- There is a utility method `apply_chat_template` that will convert from the messages list format we are familiar with, into the right input prompt for this model.
- The message in our Chat prompts gets converted
>- into a sequence of words with special tags that separate the System, user and assistant prompts
>- then the words are broken down into fragments (tokens)
>- then the tokens are replaced with token IDs
- So the input to the LLM is a sequence of token IDs

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of Pakistan?"},
    {"role": "assistant", "content": "The capital of Pakistan is Islamabad."},
    {"role": "user", "content": "And what is the currency used there?"}
]

#### For LLaMA-Instruct

In [13]:
# For LLaMA-Instruct¶
import transformers
llama_tokenizer = transformers.AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct')
prompt = llama_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

prompt = llama_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of Pakistan?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of Pakistan is Islamabad.<|eot_id|><|start_header_id|>user<|end_header_id|>

And what is the currency used there?<|eot_id|><|start_header_id|>assistant<|end_header_id|>


[128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 13, 128009, 128006, 882, 128007, 271, 3923, 374, 279, 6864, 315, 17076, 30, 128009, 128006, 78191, 128007, 271, 791, 6864, 315, 17076, 374, 92532, 13, 128009, 128006, 882, 128007, 271, 3112, 1148, 374, 279, 11667, 1511, 1070, 30, 128009, 128006, 78191, 128007, 271]


#### For Phi-Instruct

In [14]:
# For Phi-Instruct
import transformers
phi_tokenizer = transformers.AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
prompt = phi_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

prompt = phi_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
print(prompt)

<|system|>
You are a helpful assistant.<|end|>
<|user|>
What is the capital of Pakistan?<|end|>
<|assistant|>
The capital of Pakistan is Islamabad.<|end|>
<|user|>
And what is the currency used there?<|end|>
<|assistant|>

[32006, 887, 526, 263, 8444, 20255, 29889, 32007, 32010, 1724, 338, 278, 7483, 310, 21215, 29973, 32007, 32001, 450, 7483, 310, 21215, 338, 16427, 20143, 29889, 32007, 32010, 1126, 825, 338, 278, 27550, 1304, 727, 29973, 32007, 32001]


#### For Qwen-Instruct

In [15]:
# For Qwen-Instruct
import transformers
qwen_tokenizer = transformers.AutoTokenizer.from_pretrained('Qwen/Qwen2-7B-Instruct')
prompt = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

prompt = qwen_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of Pakistan?<|im_end|>
<|im_start|>assistant
The capital of Pakistan is Islamabad.<|im_end|>
<|im_start|>user
And what is the currency used there?<|im_end|>
<|im_start|>assistant

[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 3838, 374, 279, 6722, 315, 16663, 30, 151645, 198, 151644, 77091, 198, 785, 6722, 315, 16663, 374, 91432, 13, 151645, 198, 151644, 872, 198, 3036, 1128, 374, 279, 11413, 1483, 1052, 30, 151645, 198, 151644, 77091, 198]


# <span style='background :lightgreen' >7. Comparing Trained LLM Tokenizers</span>

- Let us check how different tokenizers perform tokenization of following text:
```python
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""
```
- Note down how different tokenizers deal with:
    - Capitalization.
    - Languages other than English.
    - Emojis.Programming code with keywords and whitespaces often used for indentation (in languages like Python for example).
    - Numbers and digits.
    - Special tokens. These are unique tokens that have a role other than representing text. They include tokens that indicate the beginning of the text, or the end of the text (which is the way the model signals to the system that it has completed this generation), or other functions as we’ll see.

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""


# A list of colors in RGB for representing the tokens
colors = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]


def show_tokens(sentence: str, tokenizer_name: str):
# Load the tokenizer and tokenize the input
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
# Extract vocabulary length
    print(f"Vocab length: {len(tokenizer)}")
# Print colored tokens
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors[idx % len(colors)]}m' +
            tokenizer.decode([t]) +      # small fix: decode expects a list
            '\x1b[0m',
            end=' '
        )
    # Move to next line
    print("\n")
    # Print corresponding token IDs
    print("Token IDs:")
    for t in token_ids:
        print(t, end=' ')

## a. BERT base model (uncased) (2018)
- **Tokenization method:** WordPiece, introduced in “Japanese and Korean voice search”:
- **Vocabulary size:** 30,522
- BERT was released in two major flavors: cased (where the capitalization is kept) and uncased (where all capital letters are first turned into small cap letters).

In [2]:
show_tokens(text, "bert-base-uncased")

Vocab length: 30522
[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

Token IDs:
101 2394 1998 3007 3989 100 100 2265 1035 19204 2015 6270 3904 12005 2546 1027 1027 1028 1027 2842 1024 2048 21628 2015 1024 1000 1000 2093 21628 2015 1024 1000 1000 2260 1012 1014 1008 2753 1027 5174 102 

<font color=purple>**Points to Note:**
<font color=purple><ul>Model added the special token `[CLS]` in the beginning and added the special token `[SEP]` at the end.</ul>
<font color=purple><ul>Since it is `uncased` model, so we lost all the capitalization.</ul>
<font color=purple><ul>The word  "CAPITALIZATION" is represented in in two tokens.</ul>
<font color=purple><ul>We lost the newlines.</ul>
<font color=purple><ul>The music emoji and Chinese symbol are replaced by special token `[UNK]` (lost that information).</ul>

In [3]:
show_tokens(text, "bert-base-cased")

Vocab length: 28996
[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

Token IDs:
101 1483 1105 8784 23203 9159 2162 2240 5301 13821 24805 100 100 1437 168 22559 1116 143 7264 1162 7330 8468 8914 134 134 135 134 1950 131 1160 27629 4832 131 107 107 2677 27629 4832 131 107 107 1367 119 121 115 1851 134 4372 102 

<font color=purple>**Points to Note:**
<font color=purple><ul>Since it is `cased` model, so we DONOT lost the capitalization.</ul>
<font color=purple><ul>The word  "CAPITALIZATION" is represented in in eight tokens.</ul>
<font color=purple><ul>The word  "False" is also broken down in three tokens.</ul>
<font color=purple><ul>We lost the newlines.</ul>

## b. GPT-2 (2019)
- **Tokenization method:** Byte pair encoding (BPE), introduced in “Neural machine translation of rare words with subword units”.
- **Vocabulary size:** 50,257

In [4]:
show_tokens(text, "gpt2")

Vocab length: 50257

 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

Token IDs:
198 15823 290 20176 40579 14887 6234 198 8582 236 113 16268 116 253 198 12860 62 83 482 641 10352 6045 1288 361 6624 18189 2073 25 734 22524 11097 220 220 220 366 7683 22524 25 366 220 220 220 220 220 220 366 198 1065 13 15 9 1120 28 8054 198 

In [5]:
show_tokens("       ", "gpt2")

Vocab length: 50257
              

Token IDs:
220 220 220 220 220 220 220 

<font color=purple>**Points to Note:**
<font color=purple><ul>The newline breaks are represented in the tokenizer having TID of 198.</ul>
<font color=purple><ul>Capitalization is preserved, and the word “CAPITALIZATION” is represented in four tokens.</ul>
<font color=purple><ul>The Emoji and the Chinese symbol (🎵 鸟) are represented by multiple tokens each and the information is not lost. </ul>
<font color=purple><ul>For example, the emoji is broken down into the tokens with token IDs 8582, 236, and 113. </ul>
<font color=purple><ul>The four spaces are represented as three tokens (having TID of 220) with the final space being a part of the token for the closing quote character..</ul>

## c. Flan-T5 (2022)
- **Tokenization method:** Flan-T5 uses a tokenizer implementation called SentencePiece, introduced in “SentencePiece: A simple and language independent subword tokenizer and detokenizer for neural text processing”,
which supports BPE and the unigram language model (described in “Subword regularization: Improving neural network translation models with multiple subword candidates”).
- **Vocabulary size:** 32,100

In [6]:
show_tokens(text, "google/flan-t5-small")

Vocab length: 32100
English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600  </s> 

Token IDs:
1566 11 3087 4111 16359 20091 8015 3 2 3 2 504 834 235 2217 7 10747 7 15 14794 3 15 40 99 3274 2423 2490 2423 1307 10 192 3808 7 10 121 96 5245 3808 7 10 96 96 8013 632 1935 1752 2423 6007 3 1 

<font color=purple>**Points to Note:**
<font color=purple><ul>We lost the newlines and even the whitespace tokens.</ul>
<font color=purple><ul>Capitalization is preserved, and the word “CAPITALIZATION” is represented in five tokens.</ul>
<font color=purple><ul>The Emoji and the Chinese symbol (🎵 鸟) are lost and represented by <unk> token. </ul>
<font color=purple><ul>The False word is also divided into three tokens and the elif is also divided into two.</ul>

## d. GPT-4 (2023)
- **Tokenization method:** Byte pair encoding (BPE)
- **Vocabulary size:** A little over 100,000

In [7]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

Vocab length: 100263

 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         "
 12 . 0 * 50 = 600 
 

Token IDs:
198 23392 323 78387 50125 198 9468 236 113 18630 116 253 198 3528 29938 3641 2290 4508 624 2669 775 25 1403 23204 3047 262 330 14853 23204 25 330 996 6360 717 13 15 9 1135 28 5067 198 

<font color=purple>**Points to Note:**
<font color=purple><ul>Represents the four spaces as a single token, while this was not the case with gpt2.</ul>
<font color=purple><ul>Capitalization is preserved, and the word “CAPITALIZATION” is represented in two tokens.</ul>
<font color=purple><ul>The Emoji and the Chinese symbol (🎵 鸟) are represented by multiple tokens each and the information is not lost. </ul>
<font color=purple><ul>The False, elif, and else words are not divided into subtokens.</ul>

In [8]:
show_tokens("       ", "Xenova/gpt-4")

Vocab length: 100263
        

Token IDs:
286 

In [9]:
show_tokens("       ", "gpt2")

Vocab length: 50257
              

Token IDs:
220 220 220 220 220 220 220 

## e. StarCoder2
- **Tokenization method:** Byte pair encoding (BPE)
- **Vocabulary size:** 49,152

In [10]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

Vocab length: 49152

 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

Token IDs:
222 23543 480 38657 26965 222 3822 260 151 244 32589 277 222 2276 100 8433 3208 1686 4378 630 2394 832 63 3161 16992 1941 283 332 28033 16992 63 332 981 332 222 54 55 51 53 47 58 53 66 59 53 53 222 

<font color=purple>**Points to Note:**
<font color=purple><ul>A major difference here to everything we’ve seen so far is that each digit is assigned its own token.</ul>
<font color=purple><ul>The hypothesis here is that this would lead to better representation of numbers and mathematics.</ul>

In [11]:
show_tokens("12.0*50=600", "bigcode/starcoder2-15b")

Vocab length: 49152
1 2 . 0 * 5 0 = 6 0 0 

Token IDs:
54 55 51 53 47 58 53 66 59 53 53 

In [12]:
show_tokens("12.0*50=600", "Xenova/gpt-4")

Vocab length: 100263
12 . 0 * 50 = 600 

Token IDs:
717 13 15 9 1135 28 5067 

## f. Galactica
- The Galactica model described in “Galactica: A large language model for science” is focused on scientific knowledge and is trained on many scientific papers, reference materials, and knowledge bases. It pays extra attention to tokenization that makes it more sensitive to the nuances of the dataset it’s representing. For example, it includes special tokens for citations, reasoning, mathematics, amino acid sequences, and DNA sequences.
- **Tokenization method:** Byte pair encoding (BPE)
- - **Vocabulary size:** 50,000
- **Notice the following:** The GPT-4 tokenizer behaves similarly to its ancestor, the GPT-2 tokenizer. Some differences are:
    - The Galactica tokenizer behaves similar to StarCoder2 in that it has code in mind. It also encodes whitespaces in the same way: assigning a single token to sequences of whitespace of different lengths. It differs in that it also does that for tabs, though. So from all the tokenizers we’ve seen so far, it’s the only one that assigns a single token to the string made up of two tabs ('\t\t').

In [27]:
show_tokens(text, "facebook/galactica-1.3b")

Vocab length: 50000

 English  and  CAP ITAL IZATION 
 � � � �  � � � 
 show _ tokens  False  None  elif   ==   > =  else :  two  t abs : "      "  Three  t abs :   "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

Token IDs:
221 28003 312 18320 36247 29196 221 195 276 259 136 43089 139 276 221 14592 85 24948 12051 4830 14155 243 1414 243 52 51 3683 48 682 279 6510 48 24 344 24 5753 279 6510 48 243 24 863 24 221 39 40 36 38 32 43 38 51 44 38 38 221 

## g. Phi3
- **Tokenization method:** Byte pair encoding (BPE)
- - **Vocabulary size:** 32,000

In [28]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

Vocab length: 32011
 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

Token IDs:
29871 13 24636 322 315 3301 1806 1964 26664 8098 13 243 162 145 184 29871 236 187 162 13 4294 29918 517 12360 7700 6213 25342 1275 6736 1683 29901 1023 18859 6160 1678 376 12753 18859 29901 376 539 376 13 29896 29906 29889 29900 29930 29945 29900 29922 29953 29900 29900 13 